#
# Universidad EAFIT 
# 2026-1
# SI7016 - NLP - Lecture 07 - Prompt Engineering
#

# Ejemplos de Prompt Engineering

## 1. Zero-shot Prompting (Instrucción Directa)
Se utiliza cuando la tarea es clara y no requiere ejemplos previos para que el modelo entienda el formato.

***Ejemplo:***

***Prompt:*** "Resume el siguiente artículo sobre ética en IA en un solo párrafo de máximo 50 palabras: [Texto del artículo]".

***Aplicación:*** Ideal para tareas de síntesis rápida donde el modelo ya tiene una base de conocimiento general.

## 2. One-shot y Few-shot Prompting (Aprendizaje en Contexto)

Estas técnicas inyectan ejemplos de entrada/salida directamente en el prompt para que el modelo aprenda el patrón durante la inferencia (in-context learning).

Ejemplo de Clasificación (One-shot):

***System:*** "Clasifica reseñas de restaurantes como POSITIVA o NEGATIVA."

***User:*** "Reseña: 'El servicio fue excelente y la comida deliciosa.' Clasificación: POSITIVA. Ahora clasifica esta reseña: 'La comida estaba fría y el mesero fue grosero.'".

Ejemplo de Extracción (Few-shot):

***Prompt:***

"Entrada: Alice nació en 1990. -> Salida: {'nombre': 'Alice', 'año': 1990}"

"Entrada: Bob tiene 25 años y vive en Cali. -> Salida: {'nombre': 'Bob', 'edad': 25, 'ciudad': 'Cali'}"

"Entrada: Carlos es de Medellín y nació en 1985. -> Salida:".

In [ ]:
!pip install peft
!pip install langchain-openai

In [ ]:
import os
from langchain_openai import ChatOpenAI

# 1. Configura tu API Key (puedes usar variables de entorno)
os.environ["OPENAI_API_KEY"] = "sk-xxxx"

# 2. Instancia el modelo de chat (puedes usar gpt-4o o gpt-3.5-turbo)
model = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

# 1. Definimos los ejemplos (pares Entrada/Salida)
examples = [
    {
        "input": "¿Cómo funciona un motor de combustión interna?",
        "output": "Utiliza la explosión de combustible para mover pistones y generar movimiento mecánico."
    },
    {
        "input": "¿Cómo funciona un motor eléctrico?",
        "output": "Utiliza campos magnéticos generados por electricidad para rotar un eje y producir torque."
    }
]

# 2. Creamos el template para formatear cada ejemplo
example_formatter_template = """
Entrada: {input}
Salida: {output}\n"""

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template=example_formatter_template,
)

# 3. Configuramos el FewShotPromptTemplate
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Sigue el patrón de los siguientes ejemplos para explicar el funcionamiento de sistemas:",
    suffix="Entrada: {question}\nSalida:",
    input_variables=["question"],
    example_separator="\n",
)

# 4. Crea la cadena de ejecución
# Unimos el prompt con el modelo usando el operador pipe (|)
chain = few_shot_prompt | model

# 5. Ejecuta y obtén la respuesta real
respuesta = chain.invoke({"question": "¿Cómo funciona una turbina eólica?"})

# Imprimimos solo el contenido del mensaje
print(respuesta.content)

## 3. Chain-of-Thought (CoT - Razonamiento paso a paso)

Esta técnica es crucial para mejorar el desempeño en tareas de lógica, matemáticas y razonamiento complejo al obligar al modelo a generar pasos intermedios.

Ejemplo de Lógica Matemática:

***Prompt:*** "Juan tiene 3 manzanas, compra 5 más y da 2 a su amigo. ¿Cuántas tiene? Razona paso a paso antes de dar tu respuesta final.".

***Resultado esperado:*** El modelo desglosará: "1. Juan empieza con 3 manzanas. 2. Compra 5, tiene 8. 3. Regala 2, le quedan 6.".


In [ ]:
# Definimos un template que fuerza el razonamiento intermedio
cot_template = """
[Rol]: Experto en Procesamiento de Lenguaje Natural
[Tarea]: Responder la pregunta técnica del estudiante.
[Instrucción]: Antes de dar la respuesta final, desglosa tu razonamiento paso a paso. 

Pregunta del estudiante: {question}

Razonamiento lógico:
1.
"""

cot_prompt_template = PromptTemplate(
    input_variables=["question"],
    template=cot_template,
)

# Ejemplo de uso con el problema matemático de las manzanas o uno técnico
print(cot_prompt_template.format(
    question="Si un corpus tiene 1000 tokens y 200 son únicos, y añado un documento de 50 tokens donde 10 son palabras nuevas, ¿cuál es el nuevo tamaño del vocabulario y por qué?"
))

# 4. Role Prompting (Asignación de Roles)

Asignar una identidad específica al modelo ayuda a controlar el tono, el estilo y la profundidad técnica de la respuesta.

Ejemplo Pedagógico:

Prompt: "Actúa como un profesor de historia. Explica la Revolución Francesa a estudiantes de secundaria utilizando 5 puntos clave y un lenguaje accesible".

Ejemplo Técnico:

Prompt: "Actúa como un experto en ciberseguridad. Revisa el siguiente fragmento de código Python y señala posibles vulnerabilidades de inyección SQL."

## 5. Técnicas Avanzadas y Automáticas

Para un nivel de maestría, es importante mencionar que el prompting ha evolucionado de ser puramente manual a procesos optimizados:

***Prompt Paraphrasing:*** Consiste en generar múltiples variantes de una instrucción (usando sinónimos o reestructuraciones) para evaluar cuál ofrece el mejor rendimiento (accuracy).

***Negative Prompting:*** Instrucciones sobre lo que el modelo no debe hacer (ej. "No uses tecnicismos legales", "No menciones marcas de la competencia").

***Prompt Tuning (Soft Prompts):*** A diferencia del prompting manual, aquí se aprenden vectores de embeddings optimizados directamente en el espacio vectorial del modelo, lo que es mucho más eficiente que el fine-tuning completo

## 6. Casos de Uso Sintéticos (basados en GPT

Ejemplos de las pruebas originales de OpenAI para mostrar la versatilidad de los modelos:

***Corrección gramatical:*** "Poor English input: I eated the purple berries. Good English output: I ate the purple berries".

***Traducción con ejemplos:*** "sea otter => loutre de mer", "cheese =>".

***Unscrambling (Descifrar palabras):*** "Please unscramble the letters into a word: skicts = sticks"

In [ ]:
from huggingface_hub import login
# tu token
login("hf_xxxxxxxxxxxxx")

In [ ]:
from langchain.prompts import PromptTemplate

template = """
[Rol]: {role}
[Tarea]: {task}
[Formato]: {format}

Pregunta del estudiante: {question}
"""

prompt = PromptTemplate(
    input_variables=["role", "task", "format", "question"],
    template=template,
)

print(prompt.format(
    role="Profesor de NLP",
    task="Explica conceptos de modelos de lenguaje a estudiantes de maestría",
    format="Usa ejemplos en Python y diagramas cuando sea posible",
    question="¿Qué es un modelo de lenguaje n-grama?"
))



In [ ]:
# Ejemplo Práctico en Python (Usando la librería peft de Hugging Face)
# Aunque los extractos no incluyen el código completo del notebook, la técnica estándar para implementar el Prompt Tuning descrito en las fuentes (como el método de Lester et al., 2021) se realiza de la siguiente manera en Python:

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType
import torch

# 1. Configuración del modelo base (manteniéndolo "congelado")
model_name = "bigscience/bloom-560m" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Configurar el Prompt Tuning (Soft Prompting)
# Aquí definimos que el prompt se inicializará con un texto base [2]
config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,  # Número de vectores entrenables (soft prompts)
    prompt_tuning_init_text="Clasifica el sentimiento de este texto:",
    tokenizer_name_or_path=model_name,
)

# 3. Aplicar la configuración al modelo
# Solo los 'virtual tokens' serán entrenables; el resto del modelo queda frozen [3]
model = get_peft_model(model, config)

# 4. Verificar parámetros entrenables
model.print_trainable_parameters()

# A partir de aquí, se usaría un Trainer estándar de Hugging Face
# para ajustar los vectores del prompt con un dataset específico.

# ¿Qué sucede matemáticamente?

# Cuando el modelo recibe una consulta del usuario (x), el sistema concatena los vectores aprendidos (V) antes de la entrada:
# [V1, V2, ..., Vn]+[x]

# El modelo procesa esta secuencia y genera la respuesta basándose en la "esencia" de la tarea que estos vectores 
# han capturado durante su entrenamiento

In [ ]:
# Ejemplo Práctico en Python (Usando la librería peft)
# Siguiendo los conceptos de Li & Liang (2021) citados en tus fuentes, aquí tienes una implementación concreta:

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PrefixTuningConfig, get_peft_model, TaskType
import torch

# 1. Cargar el modelo base (ejemplo con BLOOM, un modelo autoregresivo)
model_name = "bigscience/bloom-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Configurar el Prefix Tuning
# A diferencia de Prompt Tuning, aquí los virtual_tokens se inyectan en cada capa
config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,           # Número de vectores de prefijo por capa
    prefix_projection=True,          # Recomendado para estabilizar el entrenamiento inicial
)

# 3. Aplicar Prefix Tuning al modelo
# Los parámetros originales del modelo se congelan automáticamente
model = get_peft_model(model, config)

# 4. Verificar qué parámetros se van a entrenar
# Solo se entrenarán los "prefijos" añadidos a las capas de atención
model.print_trainable_parameters()

# --- Ejemplo de Inferencia ---
inputs = tokenizer("Explica la importancia del NLP en: ", return_tensors="pt")
with torch.no_grad():
    outputs = model.generate(input_ids=inputs["input_ids"], max_new_tokens=30)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

# prompt tuning vs prefix tuning
# Arquitectura: En Prompt Tuning, los vectores solo "empujan" la entrada inicial. 
# En el Prefix Tuning, los vectores actúan como una guía constante que el modelo consulta en cada capa del Transformer durante la generación.

#Uso: Es particularmente potente para tareas de generación de texto (como resúmenes o traducción) porque condiciona el comportamiento del modelo
#  de manera más profunda que un simple prompt en la entrada.
# Eficiencia: Permite que un solo modelo base "congelado" sirva para múltiples tareas simultáneamente; solo necesitas guardar los pequeños archivos de prefijos (unos pocos megabytes) para cada tarea específica



In [ ]:
# Ejemplo conceptual: Identificación de tokens mediante gradientes
# Este enfoque se basa en la lógica de AutoPrompt (Shin et al., 2020), 
# donde se busca el token que "empuje" más fuerte la predicción hacia la etiqueta correcta

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

# 1. Configuración del modelo (usamos BERT para el ejemplo del [MASK]) [1]
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# 2. Definir la tarea: Clasificación de sentimiento [1]
# Queremos optimizar un "trigger token" para la frase: "{trigger} This movie is amazing."
text = "[MASK] This movie is amazing." 
label_token = "positive" # La etiqueta que queremos forzar

# Preprocesamiento
inputs = tokenizer(text, return_tensors="pt")
label_id = tokenizer.convert_tokens_to_ids(label_token)
#mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[3]

# 21 mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# 3. Obtener los embeddings de entrada
# Necesitamos habilitar los gradientes en los embeddings para la optimización
input_embeddings = model.get_input_embeddings()(inputs["input_ids"]).detach()
input_embeddings.requires_grad = True

# 4. Forward pass y cálculo de la pérdida
outputs = model(inputs_embeds=input_embeddings)
logits = outputs.logits
# Calculamos la pérdida en la posición del [MASK] para la etiqueta "positive"
loss = torch.nn.functional.cross_entropy(logits[:, mask_token_index, :].view(-1, model.config.vocab_size), 
                                         torch.tensor([label_id]))

# 5. Cálculo de gradientes [1]
loss.backward()
grad = input_embeddings.grad # Este es el gradiente de la pérdida respecto a la entrada

# 6. Interpretación (Gradient-based optimization)
# El gradiente en la posición del primer token nos dice en qué dirección 
# matemática deberíamos cambiar ese embedding para que la pérdida sea 0.
trigger_grad = grad[0, 0, :] # Gradiente del primer token (nuestro candidato a trigger)

print(f"Magnitud del gradiente para el trigger: {trigger_grad.norm().item()}")
# En una implementación real (como AutoPrompt), buscaríamos en el vocabulario 
# el token cuyo embedding sea más similar a la dirección opuesta de este gradiente.
